# RAG - Chatbot Tim Legal berbasis Dokumen UU/PP (PGABL)
**Nama:** Stanley Nathanael Wijaya

Notebook ini membungkus model hasil fine-tuning (`Fine-tuning_submission_PGABL_...ipynb`) dengan
pipeline **Retrieval-Augmented Generation (RAG)** di atas 4 dokumen hukum resmi:

- PP Nomor 5 Tahun 2021
- PP Nomor 35 Tahun 2021
- PP Nomor 51 Tahun 2023
- UU Nomor 6 Tahun 2023

**Rekomendasi environment:** Google Colab / Kaggle dengan GPU (T4 ke atas).

Struktur notebook:
1. Instalasi dependency & unduh 4 dokumen PDF
2. Load PDF + text splitting (chunk size & overlap eksplisit)
3. Embedding open-source -> Vector DB lokal (ChromaDB)
4. Load model hasil fine-tuning + prompt `{context}`/`{question}`
5. Interface sederhana (Python loop / Gradio)
6. **Skilled:** metadata enrichment & filtering + sitasi, Ensemble Retriever (BM25 + vektor), Parent-Child Retriever
7. **Advanced:** HyDE, Reranker (Cross-Encoder) + fallback DuckDuckGo Search


## 1. Instalasi Dependency & Unduh Dokumen

Versi `langchain` di-pin secara eksplisit karena rilis 1.x melakukan restrukturisasi API besar
(`EnsembleRetriever`, `ParentDocumentRetriever`, `InMemoryStore` dipindah/dihapus dari paket inti),
yang akan membuat notebook ini gagal jika memakai `langchain` versi terbaru tanpa pin.


In [1]:
%%capture
!pip install unsloth
!pip install -q "langchain==0.3.7" "langchain-community==0.3.7" \
    "langchain-huggingface==0.1.2" "langchain-text-splitters==0.3.2" \
    "chromadb==0.5.20" pypdf "sentence-transformers==3.3.1" rank_bm25 gradio \
    duckduckgo-search gdown

Instalasi di atas menimpa versi `numpy` yang sudah ter-load di proses Python yang sedang berjalan (dipakai bersamaan oleh `unsloth` dan `chromadb`/`sentence-transformers`), yang menyebabkan error `ModuleNotFoundError: No module named 'numpy.char'` saat import berikutnya. Cell di bawah me-restart runtime **satu kali secara otomatis** setelah instalasi agar `numpy` yang baru ter-load bersih, lalu lanjutkan dengan **Run all** kembali (aman dijalankan berulang, restart hanya terjadi sekali).

In [2]:
import os

_RESTART_FLAG = os.path.join(os.getcwd(), ".rag_deps_installed")

if not os.path.exists(_RESTART_FLAG):
    open(_RESTART_FLAG, "w").close()
    print("Dependency baru saja diinstal, me-restart runtime agar numpy konsisten...")
    print("Setelah runtime restart, jalankan ulang seluruh cell (Run all).")
    os.kill(os.getpid(), 9)
else:
    print("Runtime sudah pernah di-restart setelah instalasi dependency, lanjut.")


Runtime sudah pernah di-restart setelah instalasi dependency, lanjut.


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU tidak terdeteksi. Di Colab: Runtime > Change runtime type > pilih T4 GPU."
    )

CUDA available: True


In [4]:
import os

DATA_DIR = "knowledge_rag"
os.makedirs(DATA_DIR, exist_ok=True)

# Folder Google Drive resmi berisi 4 dokumen UU/PP wajib (lihat halaman kriteria submission)
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql"

if not any(f.endswith(".pdf") for f in os.listdir(DATA_DIR)):
    import gdown

    gdown.download_folder(DRIVE_FOLDER_URL, output=DATA_DIR, quiet=False, use_cookies=False)

pdf_paths = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATA_DIR)
    for f in files
    if f.lower().endswith(".pdf")
]
print(f"{len(pdf_paths)} dokumen PDF ditemukan:")
for p in pdf_paths:
    print(" -", p)
assert len(pdf_paths) == 4, "Seluruh 4 dokumen UU/PP WAJIB digunakan."

Retrieving folder contents


Processing file 1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU PP Nomor 5 Tahun 2021.pdf
Processing file 1trvqHE72Anu8MhsNvvYJHieWqbao2BCy PP Nomor 35 Tahun 2021.pdf
Processing file 1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT PP Nomor 51 Tahun 2023.pdf
Processing file 1jf5f9ZHF2tzcK7Mm7eHCkDEtVntQ-y1s UU Nomor 6 Tahun 2023.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU
To: /content/knowledge_rag/PP Nomor 5 Tahun 2021.pdf
100%|██████████| 17.1M/17.1M [00:00<00:00, 34.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1trvqHE72Anu8MhsNvvYJHieWqbao2BCy
To: /content/knowledge_rag/PP Nomor 35 Tahun 2021.pdf
100%|██████████| 2.52M/2.52M [00:00<00:00, 96.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT
To: /content/knowledge_rag/PP Nomor 51 Tahun 2023.pdf
100%|██████████| 2.77M/2.77M [00:00<00:00, 79.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1jf5f9ZHF2tzcK7Mm7eHCkDEtVntQ-y1s
To: /content/knowledge_rag/UU Nomor 6 Tahun 2023.pdf
100%|██████████| 85.4M/85.4M [00:01<00:00, 57.5MB/s]

4 dokumen PDF ditemukan:
 - knowledge_rag/UU Nomor 6 Tahun 2023.pdf
 - knowledge_rag/PP Nomor 5 Tahun 2021.pdf
 - knowledge_rag/PP Nomor 51 Tahun 2023.pdf
 - knowledge_rag/PP Nomor 35 Tahun 2021.pdf



Download completed


## 2. Load Dokumen PDF & Text Splitting

Ukuran `chunk_size` dan `chunk_overlap` ditentukan **secara eksplisit** (1000 karakter / 150
karakter overlap) agar potongan cukup besar untuk memuat konteks pasal, tapi tetap ringkas untuk
retrieval yang presisi.


In [5]:
from langchain_community.document_loaders import PyPDFLoader

raw_documents = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    docs = loader.load()
    raw_documents.extend(docs)

print(f"Total {len(raw_documents)} halaman dimuat dari {len(pdf_paths)} dokumen.")

Total 1949 halaman dimuat dari 4 dokumen.


In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(raw_documents)
print(f"Total {len(chunks)} chunk dihasilkan (chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}).")
print(chunks[0].page_content[:300])

Total 3234 chunk dihasilkan (chunk_size=1000, overlap=150).
SALINAN
PRESIDEN
NEPUBUK INDONESIA
UNDANG-UNDANG REPUBLIK INDONESIA
NOMOR 6 TAHUN 2023
TENTANG
PENETAPAN PERATURAN PEMERINTAH PENGGANTI UNDANG-UNDANG
NOMOR 2 TAHUN 2022 TENTANG CIPTA KERJA
MENJADI UNDANG-UNDANG
DENGAN RAHMAT TUHAN YANG MAHA ESA
PRESIDEN REPUBLIK INDONESIA,
Menimbang a bahwa untuk me


### Metadata Enrichment (Skilled)

Menambahkan metadata eksplisit per chunk: nama dokumen, nomor peraturan, tipe peraturan (UU/PP),
dan nomor halaman — dipakai untuk metadata filtering & sitasi jawaban.


In [7]:
import re


def enrich_metadata(chunk):
    filename = os.path.basename(chunk.metadata.get("source", "unknown.pdf"))
    doc_type = "UU" if filename.upper().startswith("UU") else "PP"
    match = re.search(r"Nomor\s+(\d+)\s+Tahun\s+(\d{4})", filename, re.IGNORECASE)
    nomor, tahun = (match.group(1), match.group(2)) if match else ("?", "?")

    chunk.metadata.update(
        {
            "document_name": filename,
            "doc_type": doc_type,
            "nomor_peraturan": nomor,
            "tahun_peraturan": tahun,
            "citation": f"{doc_type} No. {nomor}/{tahun}, hal. {chunk.metadata.get('page', 0) + 1}",
        }
    )
    return chunk


chunks = [enrich_metadata(c) for c in chunks]
chunks[0].metadata

{'source': 'knowledge_rag/UU Nomor 6 Tahun 2023.pdf',
 'page': 0,
 'document_name': 'UU Nomor 6 Tahun 2023.pdf',
 'doc_type': 'UU',
 'nomor_peraturan': '6',
 'tahun_peraturan': '2023',
 'citation': 'UU No. 6/2023, hal. 1'}

## 3. Embedding Open-Source -> Vector Database Lokal (ChromaDB)

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

from langchain_community.vectorstores import Chroma

PERSIST_DIR = "chroma_legal_db"
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="legal_docs",
)
print(f"Vector DB berisi {vectorstore._collection.count()} chunk.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector DB berisi 3234 chunk.


## 4. Load Model Hasil Fine-tuning

In [9]:
import os
from getpass import getpass

from huggingface_hub import login


def get_secret(name, prompt, optional=False):
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        return value
    value = getpass(prompt)
    if not value and not optional:
        raise ValueError(f"{name} wajib diisi.")
    return value


HF_TOKEN = get_secret("HF_TOKEN", "Masukkan Hugging Face Token: ")
login(token=HF_TOKEN)

HF_USERNAME = os.environ.get("HF_USERNAME") or input("Masukkan username Hugging Face kamu: ")
# Ganti ke *-grpo jika ingin memakai model hasil GRPO (Advanced)
FT_REPO_ID = os.environ.get("FT_REPO_ID") or f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-id"
print("Memuat model dari:", FT_REPO_ID)

Masukkan Hugging Face Token: ··········
Masukkan username Hugging Face kamu: xStyNWx
Memuat model dari: xStyNWx/qwen2.5-1.5b-legal-chatbot-id


In [10]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FT_REPO_ID,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
FastLanguageModel.for_inference(model)

RAG_SYSTEM_PROMPT = (
    "Kamu adalah asisten AI internal Tim Legal perusahaan. Jawablah HANYA berdasarkan "
    "konteks dokumen resmi yang diberikan. Jika jawaban tidak ada dalam konteks, katakan "
    "dengan jujur bahwa informasi tidak ditemukan pada dokumen. Jawab dalam Bahasa Indonesia."
)

RAG_PROMPT_TEMPLATE = """Konteks dokumen:
{context}

Pertanyaan: {question}

Jawablah pertanyaan di atas HANYA berdasarkan konteks di atas."""


def generate_answer(question, context, max_new_tokens=400):
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)
    convo = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        convo, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## 5. Retriever

**Basic:** similarity search sederhana dari ChromaDB.

**Skilled:** metadata filtering, Ensemble Retriever (BM25 keyword + vektor semantik, bobot
ditentukan eksplisit, ambil >= 5 dokumen), dan Parent-Child Retriever (child chunk kecil untuk
pencarian, parent chunk besar/halaman utuh untuk konteks LLM).


In [11]:
# --- Basic retriever ---
basic_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


def retrieve_basic(question):
    return basic_retriever.invoke(question)

In [12]:
# --- Skilled: metadata filtering ---
def retrieve_with_filter(question, doc_type=None, nomor_peraturan=None, k=5):
    where = {}
    if doc_type:
        where["doc_type"] = doc_type
    if nomor_peraturan:
        where["nomor_peraturan"] = nomor_peraturan
    filtered_retriever = vectorstore.as_retriever(
        search_kwargs={"k": k, "filter": where} if where else {"k": k}
    )
    return filtered_retriever.invoke(question)

In [13]:
# --- Skilled: Ensemble Retriever (BM25 keyword + vektor semantik), bobot eksplisit ---
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],  # 40% keyword (BM25), 60% semantik (vektor)
)

sample_docs = ensemble_retriever.invoke("upah lembur pekerja")
print(f"Ensemble retriever mengambil {len(sample_docs)} dokumen.")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Ensemble retriever mengambil 10 dokumen.


In [14]:
# --- Skilled: Parent-Child Retriever ---
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

parent_child_vectorstore = Chroma(
    collection_name="legal_docs_parent_child",
    embedding_function=embeddings,
    persist_directory="chroma_legal_db_parent_child",
)
parent_docstore = InMemoryStore()

parent_child_retriever = ParentDocumentRetriever(
    vectorstore=parent_child_vectorstore,
    docstore=parent_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
parent_child_retriever.add_documents(raw_documents)
print("Parent-Child retriever siap. Jumlah parent chunk:", len(list(parent_docstore.yield_keys())))

/tmp/ipykernel_3420/3016570295.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  parent_child_vectorstore = Chroma(
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Parent-Child retriever siap. Jumlah parent chunk: 1949


## 6. Sitasi Jawaban (Skilled)

Menyusun konteks dari chunk terpilih sekaligus daftar sitasi (`document_name`, halaman) yang
dilampirkan di akhir jawaban.


In [15]:
def build_context_and_citations(docs):
    context_parts = []
    citations = []
    for d in docs:
        context_parts.append(d.page_content)
        citation = d.metadata.get("citation", d.metadata.get("source", "unknown"))
        if citation not in citations:
            citations.append(citation)
    return "\n\n---\n\n".join(context_parts), citations


def answer_with_citation(question, retriever_fn=retrieve_basic, **retriever_kwargs):
    docs = retriever_fn(question, **retriever_kwargs) if retriever_kwargs else retriever_fn(question)
    context, citations = build_context_and_citations(docs)
    answer = generate_answer(question, context)
    citation_text = "\n".join(f"- {c}" for c in citations)
    return f"{answer}\n\n**Sumber:**\n{citation_text}"

## 7. HyDE + Reranker + Fallback DuckDuckGo (Advanced)

1. **HyDE**: LLM membuat >= 2 jawaban hipotetis (halusinasi awal) dari pertanyaan, lalu embedding
   dari jawaban-jawaban tersebut (bukan pertanyaan mentah) dipakai untuk retrieval.
2. **Reranker**: Cross-Encoder mengurutkan ulang hasil retrieval dan hanya mengambil Top-K (K=3).
3. Jika **Relevance Score Top-1** dari reranker berada di bawah threshold, sistem beralih ke
   pencarian internet (DuckDuckGo) sebagai fallback.


In [16]:
def generate_hyde_documents(question, n=2):
    hypothetical_answers = []
    for _ in range(n):
        convo = [
            {"role": "system", "content": "Jawablah pertanyaan hukum berikut secara singkat, walau tidak yakin."},
            {"role": "user", "content": question},
        ]
        inputs = tokenizer.apply_chat_template(
            convo, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        outputs = model.generate(input_ids=inputs, max_new_tokens=150, temperature=0.9, do_sample=True)
        hypothetical_answers.append(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))
    return hypothetical_answers


def retrieve_with_hyde(question, k=8):
    hyde_docs = generate_hyde_documents(question, n=2)
    combined_query = question + "\n" + "\n".join(hyde_docs)
    hyde_embedding = embeddings.embed_query(combined_query)
    return vectorstore.similarity_search_by_vector(hyde_embedding, k=k)

In [17]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL)

RELEVANCE_THRESHOLD = 0.0  # skor logit cross-encoder; sesuaikan berdasarkan observasi empiris

def rerank(question, docs, top_k=3):
    pairs = [(question, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    top1_score = float(ranked[0][1]) if ranked else -999
    return [d for d, _ in ranked[:top_k]], top1_score

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [18]:
from duckduckgo_search import DDGS


def web_search_fallback(question, max_results=3):
    with DDGS() as ddgs:
        results = list(ddgs.text(question, max_results=max_results))
    return "\n\n".join(f"{r['title']}: {r['body']}" for r in results)


def advanced_rag_answer(question):
    candidate_docs = retrieve_with_hyde(question, k=8)
    top_docs, top1_score = rerank(question, candidate_docs, top_k=3)
    print(f"[debug] Reranker Top-1 relevance score: {top1_score:.4f}")

    if top1_score < RELEVANCE_THRESHOLD:
        print("[debug] Skor di bawah threshold -> fallback ke DuckDuckGo Search")
        context = web_search_fallback(question)
        citations = ["DuckDuckGo Search (internet)"]
    else:
        context, citations = build_context_and_citations(top_docs)

    answer = generate_answer(question, context)
    citation_text = "\n".join(f"- {c}" for c in citations)
    return f"{answer}\n\n**Sumber:**\n{citation_text}"

## 8. Interface Sederhana

### Opsi A - Interactive Python Loop

In [19]:
from IPython.display import Markdown, display


def run_chat_loop(use_advanced=False):
    print("Ketik 'exit' untuk keluar.")
    while True:
        question = input("\nPertanyaan Anda: ")
        if question.strip().lower() == "exit":
            break
        if use_advanced:
            response = advanced_rag_answer(question)
        else:
            response = answer_with_citation(question, retriever_fn=lambda q: ensemble_retriever.invoke(q))
        display(Markdown(response))


# Jalankan salah satu baris berikut secara interaktif:
# run_chat_loop(use_advanced=False)
# run_chat_loop(use_advanced=True)

### Opsi B - Gradio Interface

In [20]:
import gradio as gr


def gradio_answer(question, use_advanced):
    if use_advanced:
        return advanced_rag_answer(question)
    return answer_with_citation(question, retriever_fn=lambda q: ensemble_retriever.invoke(q))


demo = gr.Interface(
    fn=gradio_answer,
    inputs=[
        gr.Textbox(label="Pertanyaan", placeholder="Tanyakan seputar UU/PP Ketenagakerjaan..."),
        gr.Checkbox(label="Gunakan pipeline Advanced (HyDE + Reranker + DuckDuckGo fallback)", value=False),
    ],
    outputs=gr.Markdown(label="Jawaban"),
    title="Chatbot Tim Legal berbasis RAG",
    description="Fine-tuned SLM + RAG di atas dokumen UU/PP Ketenagakerjaan.",
)

# demo.launch(share=True)  # uncomment untuk menjalankan UI interaktif

## 9. Uji Coba Test Case Wajib (Advanced)

Prompt: *"Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang
lembur?"*


In [21]:
test_question = (
    "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. "
    "Apakah saya berhak dapat uang lembur?"
)

# Gunakan model hasil GRPO (FT_REPO_ID diarahkan ke *-grpo) agar proses <think> muncul di jawaban.
response = advanced_rag_answer(test_question)
display(Markdown(response))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[debug] Reranker Top-1 relevance score: -6.7438
[debug] Skor di bawah threshold -> fallback ke DuckDuckGo Search


/tmp/ipykernel_3420/2986898570.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use time

Ya, Anda berhak mendapatkan tunai lembur karena melakukan tugas-tugas administratif.

**Sumber:**
- DuckDuckGo Search (internet)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag